# MaisonDeLUX data repair — model-ready v1

This notebook repairs the existing processed dataset without downloading data or modifying the source CSV. Corrections preserve original evidence in companion columns. Only confirmed duplicates and probable numeric errors are excluded; uncertain cases remain flagged.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'data' / 'processed' / 'maisondelux_clean.csv').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.src.data_repair.model_ready import (
    EXCLUDE_FROM_MODEL,
    SAFE_CANDIDATE_FEATURES,
    TARGET,
    run_repair,
)

INPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'maisondelux_clean.csv'
INPUT_PATH.relative_to(PROJECT_ROOT)

WindowsPath('data/processed/maisondelux_clean.csv')

## Repair rules

- Property type requires explicit textual evidence; weak evidence becomes `unknown`.
- Sentence-like neighborhoods become null in `neighborhood_clean`; no replacement is invented.
- Duplicate levels A/B are identifiers, level C is a structured fingerprint, and level D is tightly blocked title similarity.
- A suspicious batch is flagged before row decisions. The batch is never removed wholesale.
- Publication dates, coordinates, URLs, and native IDs are never fabricated.

In [2]:
result = run_repair(PROJECT_ROOT)
report = result.report
output_paths = {name: path.relative_to(PROJECT_ROOT) for name, path in result.paths.items()}
display(pd.Series(output_paths, name='project_relative_path').to_frame())

,project_relative_path
csv,data\processed\maisondelux_model_ready_v1.csv
parquet,data\processed\maisondelux_model_ready_v1.parquet
report_json,reports\data_quality\model_ready_v1_quality_re...
report_markdown,reports\data_quality\model_ready_v1_quality_re...
exclusions,reports\data_quality\model_ready_v1_exclusions...
duplicate_groups,reports\data_quality\model_ready_v1_duplicate_...
batch_audit,reports\data_quality\model_ready_v1_batch_audi...
outlier_audit,reports\data_quality\model_ready_v1_outlier_au...
neighborhood_rejections,reports\data_quality\model_ready_v1_neighborho...


## Final row reconciliation

In [3]:
summary = pd.Series(report['summary'], name='value').to_frame()
display(summary)

exclusions = pd.DataFrame([
    {
        'reason': reason,
        'rows': count,
        'explanation': report['exclusion_reason_explanations'][reason],
    }
    for reason, count in report['exclusion_reason_counts'].items()
])
display(exclusions)

,value
original_rows,13867
final_model_ready_rows,13537
rows_removed,330
confirmed_duplicates_removed,26
probable_parsing_errors_removed,304
possible_duplicates_retained,3455
property_type_rows_changed,1659
invalid_neighborhoods_detected,610
regions,10
cities,30


,reason,rows,explanation
0,probable_parsing_error,304,Numeric magnitude or consistency evidence indi...
1,confirmed_duplicate,26,A higher-evidence row represents the same prop...


## Property type repair

In [4]:
property_types = (
    pd.concat([
        pd.Series(report['property_types_before'], name='before'),
        pd.Series(report['property_types_after'], name='after'),
    ], axis=1)
    .fillna(0)
    .astype(int)
)
display(property_types)
display(pd.Series(report['property_type_evidence'], name='rows').to_frame())

,before,after
appartement,13867,11918
unknown,0,1247
studio,0,202
duplex,0,107
immeuble,0,32
villa,0,13
riad,0,9
maison,0,6
bureau,0,2
magasin,0,1


,rows
explicit_appartement,12208
weak_or_ambiguous_evidence,1269
explicit_studio,214
explicit_duplex,109
explicit_immeuble,33
explicit_villa,13
explicit_riad,9
explicit_maison,8
explicit_bureau,3
explicit_magasin,1


## Neighborhood quality

In [5]:
display(pd.Series({
    'valid_rows': report['neighborhood_quality']['valid_rows'],
    'invalid_rows': report['neighborhood_quality']['invalid_rows'],
}, name='rows').to_frame())
display(pd.Series(report['neighborhood_quality']['rejection_reason_counts'], name='rows').to_frame())
display(pd.DataFrame(report['neighborhood_quality']['rejected_examples']))

,rows
valid_rows,13257
invalid_rows,610


,rows
same_as_city,238
trailing_preposition,166
sentence_or_promotional_phrase|trailing_preposition,135
token_count|sentence_or_promotional_phrase|trailing_preposition,18
sentence_or_promotional_phrase,12
token_count|trailing_preposition,11
sentence_or_promotional_phrase|measurement_or_travel_time|trailing_preposition,8
sentence_punctuation,3
token_count|sentence_or_promotional_phrase,2
measurement_or_travel_time|trailing_preposition,2


,city,neighborhood_original,neighborhood_rejection_reason
0,El Jadida,El Jadida,same_as_city
1,Berrechid,Berrechid,same_as_city
2,Larache,Larache,same_as_city
3,Casablanca,l'achat à,sentence_or_promotional_phrase|trailing_prepos...
4,Marrakech,Golf City Prestigia Votre agence immobilière LYZ,sentence_or_promotional_phrase
5,Marrakech,Les Portes de,trailing_preposition
6,Marrakech,une résidence neuve sur la Route de Casablanca à,token_count|sentence_or_promotional_phrase|tra...
7,Marrakech,mosquée Koutoubia et à 15 minutes de l’aéro...,token_count|measurement_or_travel_time
8,Marrakech,le prestigieux Golf City de,trailing_preposition
9,Ifrane,prairie à,trailing_preposition


## Hidden duplicates

In [6]:
display(pd.Series(report['duplicate_quality'], name='value').to_frame())
display(result.duplicate_groups.head(30))

,value
confirmed_groups,26
confirmed_rows_removed,26
possible_groups,1600
possible_rows_retained,3455


,duplicate_group_id,duplicate_status_repaired,duplicate_match_level,duplicate_keep,listing_id,source,source_listing_id,url,city,neighborhood_clean,surface_m2,price_mad,bedrooms,bathrooms,title_raw,source_record_path
9,confirmed-090940a925a4,unique,C_exact_fingerprint_and_title_cross_batch,True,mubawab.ma:native:8143761,mubawab.ma,8143761.0,https://www.mubawab.ma/fr/a/8143761/vente-appa...,Rabat,Haut Agdal,198.0,3350000.0,3.0,2.0,Vente Appartement Rabat Agdal,data\external\recovery_archive\20260902T224619...
6195,confirmed-090940a925a4,confirmed_duplicate,C_exact_fingerprint_and_title_cross_batch,False,mubawab.ma:content:3fdad4ac01691d88dacba045,mubawab.ma,NaN,NaN,Rabat,Haut Agdal,198.0,3350000.0,3.0,2.0,Vente Appartement Rabat Agdal,git:HEAD:data/raw/maisonlux_maroc_complet.csv
18,confirmed-1e4cb27e8ea5,unique,C_exact_fingerprint_and_title_cross_batch,True,mubawab.ma:native:8258601,mubawab.ma,8258601.0,https://www.mubawab.ma/fr/a/8258601/appartemen...,Casablanca,Les princesses,184.0,2500000.0,3.0,3.0,Appartement à vendre Les princesses,data\external\recovery_archive\20260902T224619...
13532,confirmed-1e4cb27e8ea5,confirmed_duplicate,C_exact_fingerprint_and_title_cross_batch,False,mubawab.ma:content:fd07f6fe566e885026caa2cc,mubawab.ma,NaN,NaN,Casablanca,Les princesses,184.0,2500000.0,3.0,3.0,Appartement à vendre Les princesses,git:HEAD:data/raw/maisonlux_maroc_complet.csv
1,confirmed-2069b8605973,unique,C_exact_fingerprint_and_title_cross_batch,True,mubawab.ma:native:7855576,mubawab.ma,7855576.0,https://www.mubawab.ma/fr/a/7855576/vente-appa...,Rabat,Agdal,235.0,5500000.0,3.0,2.0,Vente Appartement Rabat Haut Agdal,data\external\recovery_archive\20260902T224619...
8017,confirmed-2069b8605973,confirmed_duplicate,C_exact_fingerprint_and_title_cross_batch,False,mubawab.ma:content:6d1ef2a5ed060bc67bb0264a,mubawab.ma,NaN,NaN,Rabat,Agdal,235.0,5500000.0,3.0,2.0,Vente Appartement Rabat Haut Agdal,git:HEAD:data/raw/maisonlux_maroc_complet.csv
11,confirmed-22f515c2cfcf,unique,C_exact_fingerprint_and_title_cross_batch,True,mubawab.ma:native:8159701,mubawab.ma,8159701.0,https://www.mubawab.ma/fr/a/8159701/vente-appa...,Rabat,Riyad,201.0,4500000.0,3.0,2.0,Vente Appartement Hay Riad Rabat,data\external\recovery_archive\20260902T224619...
8824,confirmed-22f515c2cfcf,confirmed_duplicate,C_exact_fingerprint_and_title_cross_batch,False,mubawab.ma:content:821e56477568c070f9552ecc,mubawab.ma,NaN,NaN,Rabat,Riyad,201.0,4500000.0,3.0,2.0,Vente Appartement Hay Riad Rabat,git:HEAD:data/raw/maisonlux_maroc_complet.csv
4,confirmed-34b37fcd2bc3,unique,C_exact_fingerprint_and_title_cross_batch,True,mubawab.ma:native:8111061,mubawab.ma,8111061.0,https://www.mubawab.ma/fr/a/8111061/vente-appa...,Rabat,Agdal,300.0,3500000.0,3.0,2.0,Vente Appartement Rabat Bas Agdal,data\external\recovery_archive\20260902T224619...
13666,confirmed-34b37fcd2bc3,confirmed_duplicate,C_exact_fingerprint_and_title_cross_batch,False,mubawab.ma:content:fff319d18c7ea2e2fd8d9466,mubawab.ma,NaN,NaN,Rabat,Agdal,300.0,3500000.0,3.0,2.0,Vente Appartement Rabat Bas Agdal,git:HEAD:data/raw/maisonlux_maroc_complet.csv


## Batch audit

In [7]:
batch_columns = [
    'batch_id', 'row_count', 'median_price_mad', 'median_surface_m2',
    'median_price_per_m2', 'p95_price_per_m2', 'p99_price_per_m2',
    'missing_title_raw_percent', 'missing_price_raw_percent',
    'missing_url_percent', 'batch_anomaly_status', 'batch_anomaly_reason',
]
display(result.batch_audit[batch_columns])

,batch_id,row_count,median_price_mad,median_surface_m2,median_price_per_m2,p95_price_per_m2,p99_price_per_m2,missing_title_raw_percent,missing_price_raw_percent,missing_url_percent,batch_anomaly_status,batch_anomaly_reason
0,git_head_complete,10109,1400000.0,100.0,13750.00,27635.856,39701.0484,0.0,0.0,100.0,not_flagged,
1,recovery_misplaced_notebook,3665,1900000.0,110.0,17078.95,108318.838,140248.2204,100.0,100.0,0.0,suspicious_high_price_per_m2_tail,The batch p95 price/m² is at least 2.5x the me...
2,recovery_dangling_v3,93,1500000.0,100.0,16972.48,32508.560,38248.3604,0.0,0.0,0.0,not_flagged,


## Numeric consistency and outlier decisions

In [8]:
display(pd.Series(report['outlier_thresholds'], name='value').to_frame())
display(pd.Series(report['outlier_decisions'], name='rows').to_frame())
display(result.outlier_audit.head(40))

,value
reference_price_per_m2_p99,39730.33
probable_batch_price_per_m2_threshold,75000.00
price_p99,10950000.00
surface_p99,342.00


,rows
not_flagged,13289
probable_error,304
uncertain,233
valid_extreme,41


,listing_id,outlier_decision,outlier_reasons,batch_id,property_type_repaired,city,surface_m2,price_mad,price_per_m2,bedrooms,bathrooms,title_raw,price_raw,url
26,mubawab.ma:native:8318564,valid_extreme,price_per_m2_above_reference_p99,recovery_dangling_v3,unknown,Rabat,156.0,7000000.0,44871.79,3.0,2.0,Penthouse meublé à vendre – l’Orangeraie Souis...,7000000.0 MAD,https://www.mubawab.ma/fr/a/8318564/penthouse-...
100,mubawab.ma:native:7427225,uncertain,price_per_m2_above_reference_p99,recovery_misplaced_notebook,appartement,El Jadida,94.0,5560000.0,59148.94,2.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/7427225/appartemen...
133,mubawab.ma:native:7720730,valid_extreme,price_above_p99|price_per_m2_above_reference_p99,recovery_misplaced_notebook,unknown,Casablanca,231.0,13500000.0,58441.56,3.0,3.0,NaN,NaN,https://www.mubawab.ma/fr/a/7720730/beau-duple...
135,mubawab.ma:native:7728790,uncertain,price_per_m2_above_reference_p99,recovery_misplaced_notebook,appartement,Berrechid,113.0,7550000.0,66814.16,3.0,2.0,NaN,NaN,https://www.mubawab.ma/fr/a/7728790/grand-appa...
144,mubawab.ma:native:7758670,probable_error,price_above_p99|price_per_m2_above_reference_p...,recovery_misplaced_notebook,appartement,Larache,100.0,12450000.0,124500.00,2.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/7758670/appartemen...
145,mubawab.ma:native:7771091,valid_extreme,surface_extreme,recovery_misplaced_notebook,villa,Casablanca,660.0,6600000.0,10000.00,3.0,3.0,NaN,NaN,https://www.mubawab.ma/fr/a/7771091/villa-%C3%...
146,mubawab.ma:native:7772397,probable_error,price_above_p99|price_per_m2_above_reference_p...,recovery_misplaced_notebook,appartement,Tanger,90.0,11200000.0,124444.44,2.0,2.0,NaN,NaN,https://www.mubawab.ma/fr/a/7772397/appartemen...
148,mubawab.ma:native:7777660,probable_error,price_per_m2_above_reference_p99|suspicious_ba...,recovery_misplaced_notebook,appartement,Casablanca,85.0,6900000.0,81176.47,2.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/7777660/appartemen...
153,mubawab.ma:native:7800150,probable_error,price_per_m2_above_reference_p99|suspicious_ba...,recovery_misplaced_notebook,unknown,Casablanca,78.0,8960000.0,114871.79,2.0,1.0,NaN,NaN,https://www.mubawab.ma/fr/a/7800150/%C3%A0-bea...
158,mubawab.ma:native:7805399,uncertain,price_above_p99|price_per_m2_above_reference_p99,recovery_misplaced_notebook,appartement,Tanger,130.0,12470000.0,95923.08,3.0,2.0,NaN,NaN,https://www.mubawab.ma/fr/a/7805399/appartemen...


## Geographic quality

In [9]:
display(pd.Series(report['geographic_coverage']['regions'], name='listings').to_frame())
city_coverage = pd.DataFrame(report['geographic_coverage']['cities'])
display(city_coverage)
display(city_coverage.loc[city_coverage['low_sample']])

,listings
Casablanca-Settat,7378
Marrakech-Safi,2558
Tanger-Tétouan-Al Hoceïma,1426
Rabat-Salé-Kénitra,1030
Fès-Meknès,564
Souss-Massa,463
L'Oriental,74
Béni Mellal-Khénifra,40
Laâyoune-Sakia El Hamra,3
Drâa-Tafilalet,1


,city,listings,percent,low_sample
0,Casablanca,6778,50.07,False
1,Marrakech,2472,18.26,False
2,Tanger,1255,9.27,False
3,Agadir,458,3.38,False
4,Rabat,400,2.95,False
5,Fès,314,2.32,False
6,Meknès,247,1.82,False
7,Mohammedia,233,1.72,False
8,Kénitra,231,1.71,False
9,Salé,205,1.51,False


,city,listings,percent,low_sample
17,Larache,27,0.20,True
18,Béni Mellal,23,0.17,True
19,Bouznika,20,0.15,True
20,Safi,18,0.13,True
21,Khouribga,17,0.13,True
22,Settat,16,0.12,True
23,Tiznit,5,0.04,True
24,Nador,5,0.04,True
25,Ifrane,3,0.02,True
26,Fnidek,3,0.02,True


## Missingness and numeric ranges

In [10]:
display(pd.Series(report['missing_percent_final'], name='missing_percent').to_frame())
display(pd.DataFrame(report['numeric_percentiles_final']).T)

,missing_percent
price_mad,0.00
surface_m2,0.00
city,0.00
region,0.00
neighborhood_clean,3.94
property_type_repaired,0.00
bedrooms,0.72
bathrooms,1.43
url,74.48
source_listing_id,77.69


,min,p01,p05,p25,p50,p75,p95,p99,max
price_mad,60000.0,300000.00,470000.0,1000000.00,1500000.0,2350000.00,4524000.0,7782000.00,34000000.00
surface_m2,14.0,42.00,52.0,79.00,105.0,142.00,234.0,342.64,2200.00
price_per_m2,1000.0,4142.86,6250.0,10461.54,14500.0,19420.29,29074.0,52382.61,149038.46


## Modeling feature audit

In [11]:
display(Markdown('**TARGET**: ' + ', '.join(f'`{item}`' for item in TARGET)))
display(Markdown('**SAFE_CANDIDATE_FEATURES**: ' + ', '.join(f'`{item}`' for item in SAFE_CANDIDATE_FEATURES)))
display(Markdown('**EXCLUDE_FROM_MODEL**: ' + ', '.join(f'`{item}`' for item in EXCLUDE_FROM_MODEL)))
display(pd.DataFrame([
    {'excluded_group': group, 'columns': ', '.join(columns)}
    for group, columns in report['feature_audit']['exclusion_rationale'].items()
]))

**TARGET**: `price_mad`

**SAFE_CANDIDATE_FEATURES**: `region`, `city`, `neighborhood_clean`, `property_type_repaired`, `surface_m2`, `bedrooms`, `bathrooms`, `furnished_status`, `parking`, `balcony`, `sea_view`

**EXCLUDE_FROM_MODEL**: `price_per_m2`, `price_per_m2_original`, `price_per_m2_recomputed`, `price_raw`, `listing_id`, `source_listing_id`, `url`, `canonical_url_repaired`, `title_raw`, `details_raw`, `location_raw`, `publication_date`, `publication_date_status`, `scraped_at`, `source_record_path`, `validation_status`, `validation_reasons`, `deduplication_status`, `duplicate_of`, `duplicate_status_repaired`, `duplicate_group_id`, `duplicate_match_level`, `duplicate_keep`, `batch_id`, `batch_anomaly_status`, `numeric_consistency_status`, `outlier_decision`, `outlier_reasons`, `repair_exclusion_reason`

,excluded_group,columns
0,target_leakage,"price_per_m2, price_per_m2_original, price_per..."
1,raw_identifiers_or_traceability,"listing_id, source_listing_id, url, canonical_..."
2,unstructured_raw_evidence,"title_raw, details_raw, location_raw, price_raw"
3,unavailable_or_temporal_provenance,"publication_date, publication_date_status, scr..."
4,quality_control_metadata,"validation_status, validation_reasons, dedupli..."


## Final validation

In [12]:
assert report['summary']['original_rows'] == report['summary']['final_model_ready_rows'] + report['summary']['rows_removed']
assert len(result.exclusions) == report['summary']['rows_removed']
assert result.model_ready['transaction_type'].eq('sale').all()
assert result.model_ready['price_mad'].gt(0).all()
assert result.model_ready['surface_m2'].gt(0).all()
assert result.model_ready[['city', 'region']].notna().all().all()
assert not result.model_ready['duplicate_status_repaired'].eq('confirmed_duplicate').any()
assert not result.model_ready['outlier_decision'].eq('probable_error').any()
assert result.model_ready['property_type_repaired'].notna().all()
assert result.model_ready.loc[result.model_ready['neighborhood_validation_status'].eq('invalid'), 'neighborhood_clean'].isna().all()
assert result.model_ready['publication_date'].isna().all()
assert result.model_ready[['latitude', 'longitude']].isna().all().all()
assert 'price_per_m2' not in SAFE_CANDIDATE_FEATURES
assert all(path.exists() for path in result.paths.values())

display(Markdown(
    f"Validated **{len(result.model_ready):,}** model-ready rows. "
    f"Run completed without modifying `{INPUT_PATH.relative_to(PROJECT_ROOT)}`."
))

Validated **13,537** model-ready rows. Run completed without modifying `data\processed\maisondelux_clean.csv`.